# GNN-BERT Music Demo

End-to-end inference demonstration for multimodal music genre prediction using GraphSAGE audio representations and DistilBERT title representations.

## 1. Project Setup

This section loads the project directory and required libraries.

In [11]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [12]:
import os

project_root = "/content/drive/MyDrive/GNN_BERT_Music"

print("Project root exists:", os.path.exists(project_root))
print("Project contents:")

if os.path.exists(project_root):
    print(os.listdir(project_root))

Project root exists: True
Project contents:
['notebooks', 'data', 'models', 'results', 'src', 'report', '.gitignore', 'README.md', 'requirements.txt', 'config.yaml']


In [4]:
!pip install -q torch-geometric

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 39.8 MB/s eta 0:00:00


In [13]:
import os

examples_dir = os.path.join(project_root, "data", "examples")

print("Examples directory exists:", os.path.exists(examples_dir))

example_files = sorted(
    [
        os.path.join(examples_dir, f)
        for f in os.listdir(examples_dir)
        if f.endswith(".pt")
    ]
)

print("Number of example graph files:", len(example_files))

for path in example_files[:5]:
    print(os.path.basename(path))

Examples directory exists: True
Number of example graph files: 24
000002_Hip-Hop_train.pt
000005_Hip-Hop_train.pt
000010_Pop_train.pt
000140_Folk_train.pt
000141_Folk_train.pt


In [14]:
import torch
import os

example_graph_path = example_files[0]

example_graph = torch.load(
    example_graph_path,
    weights_only=False
)

print("Graph file:", os.path.basename(example_graph_path))
print("Track ID:", example_graph.track_id)
print("Genre:", example_graph.genre)
print("Title:", example_graph.title)
print("Node features shape:", example_graph.x.shape)
print("Edge index shape:", example_graph.edge_index.shape)
print("Label:", example_graph.y.item())

Graph file: 000002_Hip-Hop_train.pt
Track ID: 2
Genre: Hip-Hop
Title: Food
Node features shape: torch.Size([5, 128])
Edge index shape: torch.Size([2, 12])
Label: 3


In [5]:
import os
import glob
import json
import torch
import torch.nn as nn
import torch.nn.functional as F

from torch_geometric.data import Data
from torch_geometric.nn import SAGEConv, global_mean_pool
from transformers import AutoModel

project_root = "/content/drive/MyDrive/GNN_BERT_Music"
device = torch.device("cpu")

print("Project root:", project_root)
print("Device:", device)


Project root: /content/drive/MyDrive/GNN_BERT_Music
Device: cpu


## 2. Label Mapping

The model predicts one of the eight FMA-small top-level genres used in the project.

In [6]:
genre_to_label = {
    "Electronic": 0,
    "Experimental": 1,
    "Folk": 2,
    "Hip-Hop": 3,
    "Instrumental": 4,
    "International": 5,
    "Pop": 6,
    "Rock": 7
}

label_to_genre = {
    label: genre
    for genre, label in genre_to_label.items()
}

print(label_to_genre)


{0: 'Electronic', 1: 'Experimental', 2: 'Folk', 3: 'Hip-Hop', 4: 'Instrumental', 5: 'International', 6: 'Pop', 7: 'Rock'}


## 3. Load Example Graph

A representative preprocessed graph from `data/examples/` is used for demonstration.

In [9]:
import os

project_root = "/content/drive/MyDrive/GNN_BERT_Music"
examples_dir = os.path.join(project_root, "data", "examples")

print("Project root exists:", os.path.exists(project_root))
print("Examples directory:", examples_dir)
print("Examples directory exists:", os.path.exists(examples_dir))

if os.path.exists(examples_dir):
    example_files = sorted(
        [
            os.path.join(examples_dir, f)
            for f in os.listdir(examples_dir)
            if f.endswith(".pt")
        ]
    )

    print("Number of example graph files:", len(example_files))
    print("First 5 files:")
    for path in example_files[:5]:
        print(path)

Project root exists: False
Examples directory: /content/drive/MyDrive/GNN_BERT_Music/data/examples
Examples directory exists: False


In [20]:
example_files = sorted(
    glob.glob(
        os.path.join(
            project_root,
            "data",
            "examples",
            "*.pt"
        )
    )
)

if not example_files:
    raise FileNotFoundError("No example graph files were found.")

example_graph_path = example_files[0]
graph = torch.load(example_graph_path, weights_only=False)

print("Graph file:", os.path.basename(example_graph_path))
print("Track ID:", graph.track_id)
print("Title:", graph.title)
print("Genre:", graph.genre)
print("Node features:", tuple(graph.x.shape))
print("Edges:", tuple(graph.edge_index.shape))


Graph file: 000002_Hip-Hop_train.pt
Track ID: 2
Title: Food
Genre: Hip-Hop
Node features: (5, 128)
Edges: (2, 12)


## 4. Load BERT Representation

The cached DistilBERT representation corresponding to the selected track is loaded.

In [17]:
bert_embedding_dir = os.path.join(
    project_root,
    "data",
    "processed",
    "bert_embeddings"
)

train_embeddings_path = os.path.join(
    bert_embedding_dir,
    "train_embeddings.pt"
)

if not os.path.exists(train_embeddings_path):
    raise FileNotFoundError(
        "Cached BERT embeddings were not found."
    )

train_embeddings_data = torch.load(
    train_embeddings_path,
    weights_only=False
)

print("BERT embeddings loaded successfully.")
print(
    "Embedding shape:",
    tuple(train_embeddings_data["embeddings"].shape)
)
print(
    "Track ID shape:",
    tuple(train_embeddings_data["track_ids"].shape)
)


BERT embeddings loaded successfully.
Embedding shape: (6400, 768)
Track ID shape: (6400,)


## 5. Cross-Attention GNN+BERT Model

The final multimodal model combines GraphSAGE audio features with DistilBERT representations using cross-attention.

In [15]:
import torch
import torch.nn as nn
import torch.nn.functional as F

from torch_geometric.nn import SAGEConv, global_mean_pool
from torch_geometric.utils import to_dense_batch


class GNNBERTCrossAttention(nn.Module):
    def __init__(
        self,
        audio_input_dim=128,
        gnn_hidden_dim=128,
        bert_dim=768,
        num_heads=4,
        num_classes=8,
        dropout=0.3
    ):
        super().__init__()

        self.conv1 = SAGEConv(audio_input_dim, gnn_hidden_dim)
        self.conv2 = SAGEConv(gnn_hidden_dim, gnn_hidden_dim)

        self.bert_projection = nn.Linear(
            bert_dim,
            gnn_hidden_dim
        )

        self.cross_attention = nn.MultiheadAttention(
            embed_dim=gnn_hidden_dim,
            num_heads=num_heads,
            dropout=dropout,
            batch_first=True
        )

        self.attention_norm = nn.LayerNorm(
            gnn_hidden_dim
        )

        self.classifier = nn.Sequential(
            nn.Linear(gnn_hidden_dim * 2, 128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, num_classes)
        )

    def forward(
        self,
        x,
        edge_index,
        batch,
        bert_embedding
    ):
        x = self.conv1(x, edge_index)
        x = F.relu(x)

        x = F.dropout(
            x,
            p=0.3,
            training=self.training
        )

        x = self.conv2(x, edge_index)
        x = F.relu(x)

        audio_dense, node_mask = to_dense_batch(
            x,
            batch
        )

        text_features = self.bert_projection(
            bert_embedding
        )

        text_query = text_features.unsqueeze(1)

        attended_text, attention_weights = self.cross_attention(
            query=text_query,
            key=audio_dense,
            value=audio_dense,
            key_padding_mask=~node_mask
        )

        attended_text = attended_text.squeeze(1)

        attended_text = self.attention_norm(
            attended_text + text_features
        )

        audio_global = global_mean_pool(
            x,
            batch
        )

        fused_features = torch.cat(
            [
                attended_text,
                audio_global
            ],
            dim=1
        )

        logits = self.classifier(
            fused_features
        )

        return logits, attention_weights

## 6. Load Model Checkpoint

The trained cross-attention checkpoint is loaded for inference.

In [18]:
checkpoint_path = os.path.join(
    project_root,
    "results",
    "best_gnn_bert_fusion.pt"
)

if not os.path.exists(checkpoint_path):
    raise FileNotFoundError(
        "Cross-attention model checkpoint was not found."
    )

model = GNNBERTCrossAttention().to(device)

checkpoint = torch.load(
    checkpoint_path,
    map_location=device,
    weights_only=False
)

if isinstance(checkpoint, dict) and "model_state_dict" in checkpoint:
    model.load_state_dict(
        checkpoint["model_state_dict"]
    )
else:
    model.load_state_dict(checkpoint)

model.eval()

print("Model checkpoint loaded successfully.")


Model checkpoint loaded successfully.


## 7. End-to-End Inference

The selected example graph and its text representation are passed through the multimodal model.

In [21]:
graph = graph.to(device)

batch = torch.zeros(
    graph.x.size(0),
    dtype=torch.long,
    device=device
)

track_id = int(graph.track_id)

print("Selected track ID:", track_id)
print("Selected title:", graph.title)
print("True genre:", graph.genre)


Selected track ID: 2
Selected title: Food
True genre: Hip-Hop


## 8. Prediction

The predicted genre and class probabilities are displayed below.

In [23]:
model.eval()

with torch.no_grad():
    logits, attention_weights = model(
        graph.x,
        graph.edge_index,
        batch,
        bert_embedding
    )

    probabilities = torch.softmax(
        logits,
        dim=1
    )

    predicted_label = probabilities.argmax(
        dim=1
    ).item()

true_label = graph.y.item()

print("True genre:", label_to_genre[true_label])
print("Predicted genre:", label_to_genre[predicted_label])
print("Prediction probabilities:")

for label, probability in enumerate(probabilities[0]):
    print(
        f"{label_to_genre[label]}: "
        f"{probability.item() * 100:.2f}%"
    )

True genre: Hip-Hop
Predicted genre: Electronic
Prediction probabilities:
Electronic: 28.52%
Experimental: 16.73%
Folk: 0.78%
Hip-Hop: 23.96%
Instrumental: 1.50%
International: 2.92%
Pop: 23.57%
Rock: 2.01%


## Demo Summary

This notebook demonstrates the intended end-to-end inference workflow for the trained multimodal GNN + DistilBERT system.